# LeetCode 1967 — Number of Strings That Appear as Substrings in Word bằng Python

**Link đề:** https://leetcode.com/problems/number-of-strings-that-appear-as-substrings-in-word/
**Độ khó:** Easy · **Chủ đề:** Array, String

```python
class Solution:
    def numOfStrings(self, patterns: List[str], word: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **substring** (chuỗi con liên tiếp) khác **subsequence** (dãy con) thế nào.
2. Tự viết thuật toán tìm chuỗi con "ngây thơ" bằng **cửa sổ trượt** `word[i:i+m]`, rồi dùng toán tử `in` có sẵn.
3. Biết cách **tiền xử lý** mọi chuỗi con của `word` vào `set` khi có nhiều truy vấn.
4. (Nâng cao) Làm quen thuật toán **KMP** tìm chuỗi con tuyến tính.
5. Trace, chứng minh, Time/Space, lỗi hay gặp (đếm trùng, đếm số lần xuất hiện…) và biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng chuỗi `patterns` và chuỗi `word`. Trả về **số chuỗi trong `patterns`** xuất hiện như một **substring** của `word`.

**Substring** là một dãy ký tự **liên tiếp** trong chuỗi.

| `patterns` | `word` | Output | Giải thích |
|---|---|:---:|---|
| `["a","abc","bc","d"]` | `"abc"` | `3` | `a`, `abc`, `bc` có trong `abc`; `d` không |
| `["a","b","c"]` | `"aaaaabbbbb"` | `2` | `a`, `b` có; `c` không |
| `["a","a","a"]` | `"ab"` | `3` | mỗi phần tử của `patterns` được đếm **riêng**, kể cả trùng nhau |

**Constraints:** `1 <= patterns.length <= 100`, `1 <= patterns[i].length <= 100`, `1 <= word.length <= 100`, chỉ chữ thường.

Hai điểm cần chú ý ngay:
- Ta đếm **số pattern** xuất hiện, **không** đếm số lần mỗi pattern xuất hiện trong `word` (ví dụ 2: `a` xuất hiện 5 lần nhưng chỉ tính 1).
- Pattern trùng nhau **vẫn** được đếm nhiều lần (ví dụ 3).

In [1]:
patterns, word = ["a", "abc", "bc", "d"], "abc"
for p in patterns:
    print(repr(p), "có trong", repr(word), "?", p in word)

'a' có trong 'abc' ? True
'abc' có trong 'abc' ? True
'bc' có trong 'abc' ? True
'd' có trong 'abc' ? False


## 2. Nền tảng Python cần dùng

### 2.1. Substring vs subsequence

- **Substring:** liên tiếp. Các substring của `"abc"`: `a, b, c, ab, bc, abc` (6 cái).
- **Subsequence:** giữ thứ tự nhưng **có thể bỏ** ký tự ở giữa: `"ac"` là subsequence của `"abc"` nhưng **không** là substring.

Một chuỗi dài `n` có `n(n+1)/2` substring (tính theo vị trí).

In [2]:
word = "abc"
subs = [word[i:j] for i in range(len(word)) for j in range(i + 1, len(word) + 1)]
print(subs, len(subs))
print("ac" in word)     # False: "ac" không phải substring

['a', 'ab', 'abc', 'b', 'bc', 'c'] 6
False


List comprehension **hai tầng** `[... for i in ... for j in ...]` tương đương hai vòng `for` lồng nhau: vòng ngoài `i` (điểm bắt đầu), vòng trong `j` (điểm kết thúc, không tính `j`).

### 2.2. Toán tử `in` với chuỗi

`p in word` trả `True` nếu `p` là substring của `word`. Python cài đặt bằng thuật toán tìm kiếm tối ưu bên trong (nhanh trong thực tế). Chuỗi rỗng `""` là substring của mọi chuỗi.

In [3]:
print("bc" in "abc", "cb" in "abc", "" in "abc")

True False True


### 2.3. `str.find` và `str.count`

- `word.find(p)` trả vị trí xuất hiện **đầu tiên** hoặc `-1`.
- `word.count(p)` đếm số lần xuất hiện **không chồng nhau** — không phải thứ đề hỏi, nhưng hay bị dùng nhầm.

In [4]:
print("aaaaabbbbb".find("b"), "aaaaabbbbb".find("c"))     # 5 -1
print("aaaaabbbbb".count("a"), "aaaa".count("aa"))        # 5 2  (không chồng nhau: aa|aa)

5 -1
5 2


### 2.4. Cửa sổ trượt (sliding window) độ dài cố định

Để kiểm tra `p` (dài `m`) có trong `word` (dài `n`) không, xét mọi **vị trí bắt đầu** `i` từ `0` tới `n - m`, so `word[i:i+m]` với `p`. Có `n - m + 1` cửa sổ.

In [5]:
word, p = "abcab", "ab"
n, m = len(word), len(p)
for i in range(n - m + 1):
    print(i, word[i:i + m], word[i:i + m] == p)

0 ab True
1 bc False
2 ca False
3 ab True


Nếu `m > n` thì `range(n - m + 1)` là `range(số âm hoặc 0)` → **không lặp lần nào** → tự động `False`. Không cần xử lý riêng.

## 3. Tính tay

Ví dụ 1: `word = "abc"` có substring `{a, b, c, ab, bc, abc}`.

| pattern | Có trong tập substring? | count |
|---|:---:|:---:|
| a | có | 1 |
| abc | có | 2 |
| bc | có | 3 |
| d | không | 3 |

Kết quả `3`.

## 4. Cách 1 — Baseline: tự cài đặt tìm chuỗi con bằng cửa sổ trượt

So **từng ký tự** để hiểu điều `in` làm bên trong.

In [6]:
def contains_naive(word, p):
    n, m = len(word), len(p)
    for i in range(n - m + 1):              # mọi vị trí bắt đầu
        j = 0
        while j < m and word[i + j] == p[j]:
            j += 1
        if j == m:                          # khớp trọn p
            return True
    return False

def num_of_strings_naive(patterns, word):
    count = 0
    for p in patterns:
        if contains_naive(word, p):
            count += 1
    return count

print(num_of_strings_naive(["a", "abc", "bc", "d"], "abc"))        # 3
print(num_of_strings_naive(["a", "b", "c"], "aaaaabbbbb"))         # 2
print(num_of_strings_naive(["a", "a", "a"], "ab"))                 # 3

3
2
3


### 4.1. Chi phí của cách ngây thơ

Với một pattern dài `m` và `word` dài `n`: tối đa `(n - m + 1)` vị trí × `m` phép so → `O(n·m)`. Ca tệ: `word = "aaaa…ab"`, `p = "aa…ab"` — mỗi vị trí so gần hết `p` rồi mới hỏng ở ký tự cuối.

Với `P` pattern: `O(P·n·m)` ≤ `100 · 100 · 100 = 10^6` — thừa sức qua đề. Nhưng ta vẫn học thêm các cách tốt hơn để dùng ở bài có ràng buộc lớn.

In [7]:
def comparisons_naive(word, p):
    n, m, cnt = len(word), len(p), 0
    for i in range(n - m + 1):
        j = 0
        while j < m:
            cnt += 1
            if word[i + j] != p[j]:
                break
            j += 1
        if j == m:
            return cnt
    return cnt

print("ca dễ :", comparisons_naive("abcdefghij" * 10, "xyz"), "phép so")
print("ca khó:", comparisons_naive("a" * 99 + "b", "a" * 49 + "b"), "phép so")

ca dễ : 98 phép so
ca khó: 2550 phép so


## 5. Lời giải chính — dùng `in` cho từng pattern

Insight đơn giản: mỗi pattern **độc lập** (gợi ý 1 của đề), và Python đã có phép kiểm tra substring (gợi ý 2). Đếm số pattern thoả bằng `sum` trên bool.

**Đoạn 1:** duyệt `p` trong `patterns`.
**Đoạn 2:** `p in word` cho `True/False`.
**Đoạn 3:** cộng dồn — `True` là 1, `False` là 0.

In [8]:
def num_of_strings(patterns, word):
    return sum(p in word for p in patterns)

print(num_of_strings(["a", "abc", "bc", "d"], "abc"), num_of_strings(["a", "b", "c"], "aaaaabbbbb"),
      num_of_strings(["a", "a", "a"], "ab"))
print(sum([True, True, False]))   # 2 — nhắc lại: bool cộng như số

3 2 3
2


## 6. Mổ xẻ: vì sao không cần xử lý trùng hay độ dài?

- **Pattern trùng:** `sum` duyệt **từng phần tử** của list, nên `["a","a","a"]` được kiểm tra 3 lần, mỗi lần +1 → đúng ví dụ 3. Nếu viết `set(patterns)` sẽ **sai**.
- **Pattern dài hơn `word`:** `p in word` tự trả `False`.
- **Pattern xuất hiện nhiều lần:** `in` chỉ trả `True` một lần → mỗi pattern góp tối đa 1.

In [9]:
print(sum(p in "ab" for p in ["a", "a", "a"]))      # 3 — đúng
print(sum(p in "ab" for p in set(["a", "a", "a"]))) # 1 — SAI nếu bỏ trùng
print("abcdef" in "abc")                            # False — pattern dài hơn word

3
1
False


## 7. TRACE từng bước

Ví dụ 2: `patterns = ["a","b","c"]`, `word = "aaaaabbbbb"`. Trace cả vị trí đầu tiên tìm thấy (dùng `find`) để thấy `in` dừng sớm ngay khi gặp.

| p | `word.find(p)` | `p in word` | count |
|---|:---:|:---:|:---:|
| a | 0 | True | 1 |
| b | 5 | True | 2 |
| c | -1 | False | 2 |

In [10]:
def trace_num_of_strings(patterns, word):
    print(f"word = {word!r}")
    count = 0
    for idx, p in enumerate(patterns):
        pos = word.find(p)
        found = pos != -1
        count += found
        where = f"tại vị trí {pos}: {word[:pos]}[{p}]{word[pos + len(p):]}" if found else "không có"
        print(f"  patterns[{idx}] = {p!r:<6} -> {where:<32} count = {count}")
    return count

print("Kết quả:", trace_num_of_strings(["a", "b", "c"], "aaaaabbbbb"))
print("Kết quả:", trace_num_of_strings(["a", "abc", "bc", "d"], "abc"))

word = 'aaaaabbbbb'
  patterns[0] = 'a'    -> tại vị trí 0: [a]aaaabbbbb       count = 1
  patterns[1] = 'b'    -> tại vị trí 5: aaaaa[b]bbbb       count = 2
  patterns[2] = 'c'    -> không có                         count = 2
Kết quả: 2
word = 'abc'
  patterns[0] = 'a'    -> tại vị trí 0: [a]bc              count = 1
  patterns[1] = 'abc'  -> tại vị trí 0: [abc]              count = 2
  patterns[2] = 'bc'   -> tại vị trí 1: a[bc]              count = 3
  patterns[3] = 'd'    -> không có                         count = 3
Kết quả: 3


### 7.1. Trace cửa sổ trượt cho một pattern

Xem cách ngây thơ tìm `"bc"` trong `"abcbc"`: thử vị trí 0 (hỏng ngay ký tự đầu), vị trí 1 khớp trọn → dừng.

In [11]:
def trace_contains(word, p):
    n, m = len(word), len(p)
    for i in range(n - m + 1):
        j = 0
        while j < m and word[i + j] == p[j]:
            j += 1
        print(f"  i={i}: cửa sổ {word[i:i + m]!r} khớp {j}/{m} ký tự")
        if j == m:
            print("  -> tìm thấy")
            return True
    print("  -> không có")
    return False

trace_contains("abcbc", "bc")
trace_contains("abcbc", "cc")

  i=0: cửa sổ 'ab' khớp 0/2 ký tự
  i=1: cửa sổ 'bc' khớp 2/2 ký tự
  -> tìm thấy
  i=0: cửa sổ 'ab' khớp 0/2 ký tự
  i=1: cửa sổ 'bc' khớp 0/2 ký tự
  i=2: cửa sổ 'cb' khớp 1/2 ký tự
  i=3: cửa sổ 'bc' khớp 0/2 ký tự
  -> không có


False

## 8. Các cách giải khác

### 8.1. Cách 3 — tiền xử lý mọi substring của `word` vào `set`

`word` dài `n ≤ 100` có tối đa `5050` substring. Đưa hết vào `set` một lần, sau đó mỗi pattern chỉ là một phép tra `O(m)` (băm chuỗi dài `m`). Có lợi khi **số pattern rất lớn** còn `word` ngắn.

In [12]:
def num_of_strings_set(patterns, word):
    n = len(word)
    all_subs = {word[i:j] for i in range(n) for j in range(i + 1, n + 1)}   # set comprehension
    return sum(p in all_subs for p in patterns)

print(num_of_strings_set(["a", "abc", "bc", "d"], "abc"), num_of_strings_set(["a", "a", "a"], "ab"))
print(len({"aaaaabbbbb"[i:j] for i in range(10) for j in range(i + 1, 11)}), "substring khác nhau của 'aaaaabbbbb'")

3 3
35 substring khác nhau của 'aaaaabbbbb'


Chi phí tiền xử lý: `O(n²)` substring, mỗi cái dài tới `n` → `O(n³)` ký tự để băm (≈ 1.7·10⁵ với `n = 100`). Bộ nhớ cũng `O(n³)` ký tự. Đây là đánh đổi: tốn bộ nhớ để tra cứu nhanh.

### 8.2. Cách 4 (nâng cao) — KMP: tìm substring trong `O(n + m)`

Ý tưởng KMP (Knuth–Morris–Pratt): khi đang khớp dở mà gặp ký tự sai, **không** lùi con trỏ trên `word`; thay vào đó dùng bảng `lps` (longest proper prefix which is also suffix) của `p` để biết có thể "giữ lại" bao nhiêu ký tự đã khớp.

`lps[i]` = độ dài tiền tố dài nhất của `p[:i+1]` đồng thời là hậu tố (không tính chính nó). Ví dụ `p = "abab"` → `lps = [0, 0, 1, 2]`.

In [13]:
def build_lps(p):
    lps = [0] * len(p)
    length = 0                         # độ dài tiền tố-hậu tố hiện tại
    for i in range(1, len(p)):
        while length > 0 and p[i] != p[length]:
            length = lps[length - 1]   # lùi về tiền tố-hậu tố ngắn hơn
        if p[i] == p[length]:
            length += 1
        lps[i] = length
    return lps

print(build_lps("abab"), build_lps("aaab"), build_lps("abcabd"))

[0, 0, 1, 2] [0, 1, 2, 0] [0, 0, 0, 1, 2, 0]


In [14]:
def contains_kmp(word, p):
    lps = build_lps(p)
    j = 0                              # số ký tự của p đang khớp
    for ch in word:
        while j > 0 and ch != p[j]:
            j = lps[j - 1]
        if ch == p[j]:
            j += 1
            if j == len(p):
                return True
    return False

def num_of_strings_kmp(patterns, word):
    return sum(contains_kmp(word, p) for p in patterns)

print(num_of_strings_kmp(["a", "abc", "bc", "d"], "abc"), num_of_strings_kmp(["a", "b", "c"], "aaaaabbbbb"))
print(contains_kmp("a" * 99 + "b", "a" * 49 + "b"))

3 2
True


Với đề này (mọi độ dài ≤ 100), KMP là "dùng dao mổ trâu giết gà" — nhưng hiểu nó giúp bạn giải các bài có `word` dài `10^5` trở lên (LeetCode 28, 459, 214…).

## 9. Code nộp LeetCode (Python 3)

In [15]:
from typing import List

class Solution:
    def numOfStrings(self, patterns: List[str], word: str) -> int:
        count = 0
        for p in patterns:
            if p in word:
                count += 1
        return count


sol = Solution()
assert sol.numOfStrings(["a", "abc", "bc", "d"], "abc") == 3
assert sol.numOfStrings(["a", "b", "c"], "aaaaabbbbb") == 2
assert sol.numOfStrings(["a", "a", "a"], "ab") == 3
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản một dòng: `return sum(p in word for p in patterns)`.

## 10. Vì sao thuật toán đúng?

- Đáp án theo định nghĩa = `|{ i : patterns[i] là substring của word }|` — đếm theo **chỉ số** `i`, nên phần tử trùng được tính riêng.
- Vòng lặp duyệt mỗi chỉ số đúng một lần, cộng `1` ⇔ `patterns[i] in word` là `True` ⇔ `patterns[i]` là substring.
- **Invariant:** sau khi xét `k` pattern đầu, `count` = số pattern trong `k` cái đầu là substring của `word`.

Với cách ngây thơ: `p` là substring ⇔ tồn tại `i` với `word[i:i+m] == p`; ta thử **mọi** `i` hợp lệ (`0 ≤ i ≤ n - m`), nên không bỏ sót; và chỉ trả `True` khi khớp trọn `m` ký tự nên không báo nhầm.

## 11. Time / Space Complexity

Gọi `P = len(patterns)`, `n = len(word)`, `m` = độ dài pattern lớn nhất.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Cửa sổ trượt tự viết | `O(P · n · m)` | `O(1)` | hiểu bản chất |
| 2. `in` (chính) | `O(P · n · m)` tệ nhất lý thuyết; thực tế nhanh hơn nhiều | `O(1)` | nên nộp |
| 3. `set` mọi substring | `O(n³ + P · m)` | `O(n³)` | lợi khi P rất lớn |
| 4. KMP | `O(P · (n + m))` | `O(m)` | tuyến tính mỗi pattern |

Với giới hạn đề (`≤ 100`), mọi cách đều chạy tức thì.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.numOfStrings(["a", "abc", "bc", "d"], "abc"), 3)
check("đề 2", sol.numOfStrings(["a", "b", "c"], "aaaaabbbbb"), 2)
check("đề 3 (trùng)", sol.numOfStrings(["a", "a", "a"], "ab"), 3)
check("pattern = word", sol.numOfStrings(["hello"], "hello"), 1)
check("pattern dài hơn word", sol.numOfStrings(["abcd"], "abc"), 0)
check("subsequence không tính", sol.numOfStrings(["ac"], "abc"), 0)
check("ở cuối word", sol.numOfStrings(["yz"], "xyz"), 1)
check("không pattern nào", sol.numOfStrings(["q", "r"], "abc"), 0)
check("chồng lấn", sol.numOfStrings(["aba"], "ababa"), 1)

PASS đề 1: 3
PASS đề 2: 2
PASS đề 3 (trùng): 3
PASS pattern = word: 1
PASS pattern dài hơn word: 0
PASS subsequence không tính: 0
PASS ở cuối word: 1
PASS không pattern nào: 0
PASS chồng lấn: 1


In [17]:
# Đối chiếu 4 cách trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1967)
funcs = [num_of_strings_naive, num_of_strings, num_of_strings_set, num_of_strings_kmp, Solution().numOfStrings]
for _ in range(1000):
    word = "".join(random.choice("ab") for _ in range(random.randint(1, 12)))
    patterns = ["".join(random.choice("ab") for _ in range(random.randint(1, 5))) for _ in range(random.randint(1, 6))]
    results = {f(patterns, word) for f in funcs}
    assert len(results) == 1, (patterns, word, results)
print("1000 bộ ngẫu nhiên: mọi cách khớp nhau")

1000 bộ ngẫu nhiên: mọi cách khớp nhau


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Đếm **số lần xuất hiện** thay vì số pattern

In [18]:
def buggy_count_occurrences(patterns, word):
    return sum(word.count(p) for p in patterns)

print("buggy:", buggy_count_occurrences(["a", "b", "c"], "aaaaabbbbb"), "| đúng:", num_of_strings(["a", "b", "c"], "aaaaabbbbb"))

buggy: 10 | đúng: 2


### Lỗi 2 — Bỏ trùng bằng `set(patterns)`

Ví dụ 3 của đề trả `3`, bản `set` trả `1` (đã minh hoạ ở mục 6).

### Lỗi 3 — Ngược vế: `word in p`

In [19]:
def buggy_reversed(patterns, word):
    return sum(word in p for p in patterns)

print("buggy:", buggy_reversed(["a", "abc", "bc", "d"], "abc"), "| đúng:", num_of_strings(["a", "abc", "bc", "d"], "abc"))

buggy: 1 | đúng: 3


`word in p` hỏi "`word` có nằm trong `p` không" — ngược ý đề. Chỉ `"abc"` thoả nên bản lỗi trả `1`.

### Lỗi 4 — Hiểu nhầm substring thành subsequence (hoặc "chứa đủ các chữ")

In [20]:
def buggy_letters_subset(patterns, word):
    return sum(set(p) <= set(word) for p in patterns)    # chỉ kiểm tra "có đủ chữ cái"

print("buggy:", buggy_letters_subset(["ac", "cba"], "abc"), "| đúng:", num_of_strings(["ac", "cba"], "abc"))

buggy: 2 | đúng: 0


`"ac"` và `"cba"` đều dùng chữ có trong `"abc"` nhưng **không** là substring (không liên tiếp / sai thứ tự).

### Lỗi 5 — Sai biên vòng cửa sổ: `range(n - m)`

Bỏ sót cửa sổ **cuối cùng** (bắt đầu tại `n - m`) → pattern nằm ở cuối `word` bị bỏ qua.

In [21]:
def buggy_window(word, p):
    n, m = len(word), len(p)
    return any(word[i:i + m] == p for i in range(n - m))     # thiếu + 1

print("buggy:", buggy_window("xyz", "yz"), "| đúng:", contains_naive("xyz", "yz"))

buggy: False | đúng: True


## 14. Mẹo nhận dạng pattern

- "Chuỗi A có xuất hiện **liên tiếp** trong B không" → `A in B` (Python), hoặc cửa sổ trượt / KMP khi cần tự cài.
- "Đếm số phần tử thoả điều kiện" → `sum(điều_kiện for x in dãy)`.
- Rất nhiều truy vấn trên **một** chuỗi ngắn → tiền xử lý tập substring (hoặc trie/suffix structure ở bài khó).
- Đọc kỹ: đếm **phần tử** hay đếm **lần xuất hiện**? Có tính **trùng** không? **substring** hay **subsequence**?
- Bài cùng họ: 28 (Find the Index of the First Occurrence), 1408 (String Matching in an Array), 392 (Is Subsequence — so sánh!), 459 (Repeated Substring Pattern).

## 15. Biến thể

### 15.1. LeetCode 1408 — từ nào là substring của một từ **khác** trong mảng?

In [22]:
def string_matching(words):
    return [w for i, w in enumerate(words) if any(i != j and w in other for j, other in enumerate(words))]

print(string_matching(["mass", "as", "hero", "superhero"]))   # ['as', 'hero']

['as', 'hero']


### 15.2. Đếm số lần xuất hiện **có chồng lấn**

`str.count` không đếm chồng lấn (`"aaaa".count("aa") == 2`). Muốn đếm chồng lấn (`3`), dùng cửa sổ trượt.

In [23]:
def count_overlapping(word, p):
    m = len(p)
    return sum(word[i:i + m] == p for i in range(len(word) - m + 1))

print("aaaa".count("aa"), count_overlapping("aaaa", "aa"))   # 2 3

2 3


### 15.3. LeetCode 392 — Is Subsequence (để phân biệt)

Hai con trỏ: duyệt `t`, tiến con trỏ trên `s` mỗi khi khớp.

In [24]:
def is_subsequence(s, t):
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)

print(is_subsequence("ac", "abc"), "ac" in "abc")   # True False — subsequence nhưng không substring

True False


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `"ab" in "cab"`, `"" in "x"`, `"aaa".count("aa")`, `"abc".find("c")`.

**Bài 2 (tính tay):** liệt kê mọi substring **khác nhau** của `"aba"`. Có bao nhiêu? Vì sao ít hơn `3·4/2 = 6`?

**Bài 3 (điền chỗ trống):** hoàn thiện tìm chuỗi con bằng cửa sổ trượt.

In [25]:
def exercise_contains(word, p):
    n, m = len(word), len(p)
    for i in range(n - m + 1):
        # TODO: nếu word[i:i+m] == p -> return True
        pass
    return False

# assert exercise_contains("abc", "bc") is True
# assert exercise_contains("abc", "ac") is False

**Bài 4 (tự viết lại):** viết `exercise_num_of_strings(patterns, word)` dùng **`exercise_contains` của bạn** (không dùng `in` giữa hai chuỗi).

In [26]:
def exercise_num_of_strings(patterns, word):
    # TODO
    pass

# assert exercise_num_of_strings(["a", "abc", "bc", "d"], "abc") == 3
# assert exercise_num_of_strings(["a", "a", "a"], "ab") == 3

**Bài 5 (biến thể):** viết `exercise_count_distinct_substrings(word)` trả số substring **khác nhau** của `word`.

**Bài 6 (mini challenge):** viết `exercise_first_occurrence(haystack, needle)` (LeetCode 28) trả chỉ số xuất hiện đầu tiên hoặc `-1`, **không** dùng `find`/`index`/`in`.

In [27]:
def exercise_count_distinct_substrings(word):
    # TODO
    pass

def exercise_first_occurrence(haystack, needle):
    # TODO
    pass

# assert exercise_count_distinct_substrings("aba") == 5
# assert exercise_count_distinct_substrings("aaa") == 3
# assert exercise_first_occurrence("sadbutsad", "sad") == 0
# assert exercise_first_occurrence("leetcode", "leeto") == -1

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `in` là substring | chuỗi rỗng luôn có | `count` không chồng lấn |
| 2 | liệt kê theo độ dài | `a, b, ab, ba, aba` | `a` xuất hiện 2 lần nhưng chỉ tính 1 |
| 3 | so cửa sổ | `word[i:i+m] == p` | `if word[i:i + m] == p: return True` |
| 4 | gọi hàm bài 3 | `sum(...)` | `sum(exercise_contains(word, p) for p in patterns)` |
| 5 | set comprehension | hai vòng `i`, `j` | `len({word[i:j] for i ... for j in range(i+1, n+1)})` |
| 6 | cửa sổ trượt trả chỉ số | vòng `i` từ 0 tới `n - m` | so từng ký tự, khớp trọn thì `return i` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `True`, `True`, `1`, `2`.

**Bài 2:** `a, b, ab, ba, aba` — 5 substring khác nhau; ít hơn 6 vì `"a"` xuất hiện ở hai vị trí (0 và 2) nhưng chỉ là **một** chuỗi.

In [28]:
print("ab" in "cab", "" in "x", "aaa".count("aa"), "abc".find("c"))
print(sorted({"aba"[i:j] for i in range(3) for j in range(i + 1, 4)}))


def answer_contains(word, p):
    n, m = len(word), len(p)
    for i in range(n - m + 1):
        if word[i:i + m] == p:
            return True
    return False


def answer_num_of_strings(patterns, word):
    return sum(answer_contains(word, p) for p in patterns)


def answer_count_distinct_substrings(word):
    n = len(word)
    return len({word[i:j] for i in range(n) for j in range(i + 1, n + 1)})


def answer_first_occurrence(haystack, needle):
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        j = 0
        while j < m and haystack[i + j] == needle[j]:
            j += 1
        if j == m:
            return i
    return -1


assert answer_contains("abc", "bc") is True
assert answer_contains("abc", "ac") is False
assert answer_num_of_strings(["a", "abc", "bc", "d"], "abc") == 3
assert answer_num_of_strings(["a", "a", "a"], "ab") == 3
assert answer_count_distinct_substrings("aba") == 5
assert answer_count_distinct_substrings("aaa") == 3
assert answer_first_occurrence("sadbutsad", "sad") == 0
assert answer_first_occurrence("leetcode", "leeto") == -1
for _ in range(300):
    h = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    nd = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    assert answer_first_occurrence(h, nd) == h.find(nd)
print("Đáp án tham khảo PASS toàn bộ test!")

True True 1 2
['a', 'ab', 'aba', 'b', 'ba']
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numOfStrings(self, patterns: List[str], word: str) -> int:
        return sum(p in word for p in patterns)    # mỗi pattern góp tối đa 1, trùng vẫn tính
```

**Câu thần chú:** *từng pattern độc lập → `in` kiểm tra substring → cộng bool.*

**Checklist trước khi submit:**
- [ ] Đếm **số pattern**, không đếm số lần xuất hiện?
- [ ] **Không** bỏ trùng `patterns`?
- [ ] Viết `p in word` (không ngược thành `word in p`)?
- [ ] Hiểu substring là **liên tiếp** (`"ac"` không có trong `"abc"`)?
- [ ] Nếu tự viết cửa sổ: vòng là `range(n - m + 1)`?